# Stability Analysis - MyBESS Input/Output

This notebook linearizes the initialized MyBESS model with one exposed input, `PRefPu`, and one observed output, `PInjRefPu_out`.

It retrieves the OpenModelica state-space matrices

```text
der(Delta x) = A * Delta x + B * Delta u
Delta y      = C * Delta x + D * Delta u
```

and uses the eigenvalues of `A` for the small-signal stability check.


In [1]:
using OMJulia
using LinearAlgebra
using DataFrames
using Printf


## User Configuration


In [2]:
DEFAULT_ROOT_DIR = "/home/clarafercas/dynawo-notebooks/OpenModelica_only_users"
ROOT_DIR = isdir(joinpath(DEFAULT_ROOT_DIR, "StabilityAnalysis")) ? DEFAULT_ROOT_DIR : abspath("..")

MODEL = "MyBESS_initialized_with_PRef_input"
MODEL_FILE_PATH = joinpath(ROOT_DIR, "StabilityAnalysis", "models", MODEL * ".mo")

DYNAWO_PKG_PATH = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

P_REF_INPUT_VALUE = "0.5"
LINEARIZATION_TIME = "2.0"
MATRIX_TOL = 1e-9

OUTPUT_DIR = joinpath(ROOT_DIR, "StabilityAnalysis", "outputs", MODEL)
BUILD_DIR = joinpath(OUTPUT_DIR, "omc-build")
mkpath(BUILD_DIR)

COMMAND_LINE_OPTIONS = "--matchingAlgorithm=PFPlusExt --indexReductionMethod=dynamicStateSelection -d=initialization,NLSanalyticJacobian,newInst"

isfile(MODEL_FILE_PATH) || error("Model file not found: $MODEL_FILE_PATH")
isfile(DYNAWO_PKG_PATH) || error("Dynawo package not found: $DYNAWO_PKG_PATH")
isfile(MODELICA_PKG_PATH) || error("Modelica package not found: $MODELICA_PKG_PATH")

println("Model file: ", MODEL_FILE_PATH)
println("PRefPu value used for the operating point: ", P_REF_INPUT_VALUE)
println("Linearization time: ", LINEARIZATION_TIME, " s")
println("Build directory: ", BUILD_DIR)


Model file: /home/clarafercas/dynawo-notebooks/OpenModelica_only_users/StabilityAnalysis/models/MyBESS_initialized_with_PRef_input.mo
PRefPu value used for the operating point: 0.5
Linearization time: 2.0 s
Build directory: /home/clarafercas/dynawo-notebooks/OpenModelica_only_users/StabilityAnalysis/outputs/MyBESS_initialized_with_PRef_input/omc-build


## Model Boundary

The local model keeps the initialized MyBESS dynamics, but exposes one input and one output at the top level:

```modelica
Modelica.Blocks.Interfaces.RealInput PRefPu(start = 0.5);
connect(PRefPu, BESS.PRefPu);

Modelica.Blocks.Interfaces.RealOutput PInjRefPu_out;
PInjRefPu_out = BESS.repcA.PInjRefPu;
```

`PRefPu` is the signal perturbed by the `B` matrix. `PInjRefPu_out` is only observed; it is not connected back into the model.


## Load and Build the Model


In [3]:
function omc_call(omc, expression; parsed = true)
    println("OMC -> ", expression)
    try
        return sendExpression(omc, expression; parsed = parsed)
    catch err
        println(sendExpression(omc, "getErrorString()"; parsed = false))
        rethrow(err)
    end
end

StudyOMC = OMJulia.OMCSession()

omc_call(StudyOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(StudyOMC, "loadModel(Complex)")
omc_call(StudyOMC, "loadModel(ModelicaServices)")
omc_call(StudyOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")

ModelicaSystem(
    StudyOMC,
    MODEL_FILE_PATH,
    MODEL;
    commandLineOptions = COMMAND_LINE_OPTIONS,
    customBuildDirectory = BUILD_DIR,
)

omc_call(StudyOMC, "checkModel($MODEL)", parsed = false)
setInputs(StudyOMC, "PRefPu=$P_REF_INPUT_VALUE")

println("Model built successfully.")
println("Inputs detected by OMJulia:")
display(getInputs(StudyOMC))


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.eeeZyOJeV5"


OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> checkModel(MyBESS_initialized_with_PRef_input)
Model built successfully.
Inputs detected by OMJulia:


Dict{Any, Any} with 1 entry:
  "PRefPu" => "0.5"

## Linearize the Model

The default `LINEARIZATION_TIME` is `2.0 s`, after the MyBESS voltage event from `1.0 s` to `1.5 s`.


In [4]:
setLinearizationOptions(
    StudyOMC,
    [
        "startTime=0.0",
        "stopTime=$LINEARIZATION_TIME",
        "stepSize=0.001",
        "tolerance=1e-6",
    ],
)

display(getLinearizationOptions(StudyOMC))

notebook_dir = pwd()
matrices = try
    linearize(StudyOMC; lintime = LINEARIZATION_TIME, verbose = false)
finally
    cd(notebook_dir)
end

A = Matrix{Float64}(Float64.(matrices[1]))
B_raw = Float64.(matrices[2])
C_raw = Float64.(matrices[3])
D_raw = Float64.(matrices[4])

states = String.(getLinearStates(StudyOMC))
inputs = String.(getLinearInputs(StudyOMC))
outputs = String.(getLinearOutputs(StudyOMC))

B = ndims(B_raw) == 1 ? reshape(B_raw, :, length(inputs)) : Matrix{Float64}(B_raw)
C = ndims(C_raw) == 1 ? reshape(C_raw, length(outputs), :) : Matrix{Float64}(C_raw)
D = ndims(D_raw) == 1 ? reshape(D_raw, length(outputs), length(inputs)) : Matrix{Float64}(D_raw)


Dict{AbstractString, AbstractString} with 4 entries:
  "startTime" => "0.0"
  "stopTime"  => "2.0"
  "stepSize"  => "0.001"
  "tolerance" => "1e-6"

  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.


1×1 Matrix{Float64}:
 0.0

## Linear Variables


In [5]:
println("Selected dynamic states")
display(DataFrame(index = 1:length(states), state = states))

println("Linear inputs")
display(DataFrame(index = 1:length(inputs), input = inputs))

println("Linear outputs")
display(DataFrame(index = 1:length(outputs), output = outputs))


Selected dynamic states


Row,index,state
,Int64,String
1,1,BESS_pll_integrator_y
2,2,BESS_pll_limIntegrator_y
3,3,BESS_reecC_firstOrder_y
4,4,BESS_reecC_firstOrder1_y
5,5,BESS_reecC_integrator_y
6,6,BESS_reecC_limPIDFreeze_I_y
7,7,BESS_reecC_rateLimFirstOrderFreeze_y
8,8,BESS_reecC_rateLimFirstOrderFreeze1_y
9,9,BESS_reecC_slewRateLimiter_y


Linear inputs


Row,index,input
,Int64,String
1,1,PRefPu


Linear outputs


Row,index,output
,Int64,String
1,1,PInjRefPu_out


## Matrix Results

`A`, `B`, `C`, and `D` are displayed below as the actual Julia matrices returned by OMJulia. The state/input/output tables above give the row and column order.


In [6]:
matrix_summary = DataFrame(
    matrix = ["A", "B", "C", "D"],
    size = ["$(size(A, 1)) x $(size(A, 2))", "$(size(B, 1)) x $(size(B, 2))", "$(size(C, 1)) x $(size(C, 2))", "$(size(D, 1)) x $(size(D, 2))"],
    rows = ["state derivatives", "state derivatives", "outputs", "outputs"],
    columns = ["states", "inputs", "states", "inputs"],
)

display(matrix_summary)


Row,matrix,size,rows,columns
,String,String,String,String
1,A,20 x 20,state derivatives,states
2,B,20 x 1,state derivatives,inputs
3,C,1 x 20,outputs,states
4,D,1 x 1,outputs,inputs


In [ ]:
println("A matrix")
display(A)

println("B matrix")
display(B)

println("C matrix")
display(C)

println("D matrix")
display(D)


## Small-Signal Modes

The sign and stability are taken from the real part of each eigenvalue. The oscillation frequency comes from the imaginary part.


In [8]:
mode_tol = 1e-8
eigenvalues = eigvals(A)
mode_rows = NamedTuple[]
oscillation_df = DataFrame(
    eigenvalue = ComplexF64[],
    real_part = Float64[],
    real_part_sign = String[],
    stability = String[],
    imag_part_rad_per_s = Float64[],
    frequency_hz = Float64[],
    period_s = Float64[],
)

for lambda in eigenvalues
    real_part = real(lambda)
    imag_part = imag(lambda)

    real_part_sign = if abs(real_part) <= mode_tol
        "zero"
    elseif real_part > 0
        "positive"
    else
        "negative"
    end

    stability = if real_part_sign == "positive"
        "unstable"
    elseif real_part_sign == "zero"
        "marginal"
    else
        "stable"
    end

    push!(mode_rows, (
        eigenvalue = lambda,
        real_part = real_part,
        real_part_sign = real_part_sign,
        stability = stability,
    ))

    if abs(imag_part) > mode_tol
        frequency_hz = abs(imag_part) / (2pi)
        push!(oscillation_df, (
            eigenvalue = lambda,
            real_part = real_part,
            real_part_sign = real_part_sign,
            stability = stability,
            imag_part_rad_per_s = imag_part,
            frequency_hz = frequency_hz,
            period_s = 1 / frequency_hz,
        ))
    end
end

mode_df = sort(DataFrame(mode_rows), :real_part, rev = true)
sort!(oscillation_df, :frequency_hz)

println("Eigenvalues")
display(mode_df)

println("Oscillatory eigenvalues")
display(oscillation_df)


Eigenvalues


Row,eigenvalue,real_part,real_part_sign,stability
,Complex…,Float64,String,String
1,0.0+0.0im,0.0,zero,marginal
2,0.0+0.0im,0.0,zero,marginal
3,0.0+0.0im,0.0,zero,marginal
4,-5.0e-7+0.0im,-5.0e-7,negative,stable
5,-9.99999e-7+0.0im,-9.99999e-7,negative,stable
6,-6.7145+0.0im,-6.7145,negative,stable
7,-9.88919-14.8637im,-9.88919,negative,stable
8,-9.88919+14.8637im,-9.88919,negative,stable
9,-20.0+0.0im,-20.0,negative,stable


Oscillatory eigenvalues


Row,eigenvalue,real_part,real_part_sign,stability,imag_part_rad_per_s,frequency_hz,period_s
,Complex…,Float64,String,String,Float64,Float64,Float64
1,-58.8242-0.0996354im,-58.8242,negative,stable,-0.0996354,0.0158575,63.0618
2,-58.8242+0.0996354im,-58.8242,negative,stable,0.0996354,0.0158575,63.0618
3,-9.88919-14.8637im,-9.88919,negative,stable,-14.8637,2.36563,0.422721
4,-9.88919+14.8637im,-9.88919,negative,stable,14.8637,2.36563,0.422721
5,-63.9339-15.9691im,-63.9339,negative,stable,-15.9691,2.54157,0.393458
6,-63.9339+15.9691im,-63.9339,negative,stable,15.9691,2.54157,0.393458


In [9]:
unstable_modes = mode_df[mode_df.stability .== "unstable", :]
marginal_modes = mode_df[mode_df.stability .== "marginal", :]
stable_modes = mode_df[mode_df.stability .== "stable", :]
max_real_part = maximum(mode_df.real_part)

result_summary = DataFrame(
    item = ["linearization time", "largest real part", "unstable eigenvalues", "marginal eigenvalues", "stable eigenvalues", "oscillatory eigenvalues", "oscillatory pairs"],
    value = [LINEARIZATION_TIME * " s", @sprintf("%.6g", max_real_part), string(nrow(unstable_modes)), string(nrow(marginal_modes)), string(nrow(stable_modes)), string(nrow(oscillation_df)), string(nrow(oscillation_df) ÷ 2)],
)

display(result_summary)

if nrow(unstable_modes) > 0
    println("Unstable eigenvalues")
    display(unstable_modes)
elseif nrow(marginal_modes) > 0
    println("No unstable eigenvalues, but marginal eigenvalues were found.")
    display(marginal_modes)
else
    println("All eigenvalues are stable at t = $LINEARIZATION_TIME s.")
end


Row,item,value
,String,String
1,linearization time,2.0 s
2,largest real part,0
3,unstable eigenvalues,0
4,marginal eigenvalues,3
5,stable eigenvalues,17
6,oscillatory eigenvalues,6
7,oscillatory pairs,3


No unstable eigenvalues, but marginal eigenvalues were found.


Row,eigenvalue,real_part,real_part_sign,stability
,Complex…,Float64,String,String
1,0.0+0.0im,0.0,zero,marginal
2,0.0+0.0im,0.0,zero,marginal
3,0.0+0.0im,0.0,zero,marginal


## Run Another Operating Point

Change `LINEARIZATION_TIME` in the configuration cell and rerun the notebook.

Useful first checks:
- `"0.0"`: before the voltage event
- `"1.2"`: during the voltage event
- `"2.0"`: after the voltage event
